# T5-base inverter on 100k one-million-instructions prompts (Llama-2-7B log-probs)

This is the scaled-up version of the Dolly run, with the same model, settings and code.

**Training data:** the vectors from the `vec2text-1m-llama2-vectors` notebook, i.e. 101k prompts
from one-million-instructions, encoded with the teammate's method. 1% of them is held out as
validation.

**Run:**
- AdamW, lr 2e-4, constant after warmup, batch 32, fp32, one T4
- `MAX_STEPS` steps, sized to fit in Kaggle's 12-hour limit
- the saved "best" model is chosen by **BLEU** (`VEC2TEXT_BEST_METRIC`), not eval loss

**Evaluation** with `eval_inverter.py`, on two sets:
- the held-out one-million-instructions validation split;
- the 737 Dolly validation prompts. They were never seen in training, which makes this a fair
  comparison with the 14k-prompt Dolly model.

**Inputs:**
- the `dolly15k-llama2-7b-logprobs` dataset;
- the outputs of the `vec2text-1m-llama2-vectors` and `vec2text-dolly-llama2-t5base` notebooks.

Accelerator: GPU T4, with internet on.

In [ ]:
SMOKE = True
MAX_STEPS = 60 if SMOKE else 30_000     # ~1.0 step/s on a T4 -> ~8.5 h incl. evals

REPO_URL = "https://github.com/Yae-sine/Language-Model-Inversion.git"
REPO_BRANCH = "main"
REPO_DIR = "/tmp/lmi"
V2T_DIR = f"{REPO_DIR}/vec2text"
NPY_DIR = "/tmp/llama2_dolly15k"          # teammate's Dolly vectors (from the dataset input)
DATASET_DIR = "/tmp/dolly_llama2_inv"     # Dolly HF dataset (same split as before)
WORK = "/kaggle/working"

OM_DATASET_DIR = "/tmp/om_llama2_inv"
OUT_DIR = f"{WORK}/saves/" + ("smoke" if SMOKE else "t5base_llama7b_1m100k")
EVAL_DIR = f"{WORK}/eval_1m"

In [ ]:
import glob, json, os, shutil, subprocess, sys, zipfile


def sh(cmd, cwd=None, env=None, log=None):
    # Run a shell command, stream its output into the notebook, fail loudly on error.
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    f = open(log, "a") if log else None
    for line in p.stdout:
        print(line, end="", flush=True)
        if f:
            f.write(line)
            f.flush()
    rc = p.wait()
    if f:
        f.close()
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")


sh("nvidia-smi")
sh(f"rm -rf {REPO_DIR} && git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} log --oneline -1")

## Environment

`transformers` is pinned to 4.45.2. That is the last release that still accepts the
`evaluation_strategy` setting vec2text's `TrainingArguments` uses (removed in 4.46). It is also the
first release that works with `tokenizers` 0.20, which ships Python 3.13 builds; older `tokenizers`
fail to install on Kaggle's Python 3.13. A constraints file keeps Kaggle's own torch build.

`torchvision` is uninstalled. When it is loaded, `datasets` 3.x's torch formatter imports
`torchvision.io.VideoReader`, which recent torchvision removed, so reading the dataset crashes.
vec2text never uses torchvision.

In [ ]:
torch_ver = subprocess.check_output(
    [sys.executable, "-c", "import torch; print(torch.__version__)"], text=True).strip()
with open("/tmp/constraints.txt", "w") as f:
    f.write(f"torch=={torch_ver}\n")
print("keeping torch", torch_ver)

sh('pip install -q -c /tmp/constraints.txt "transformers==4.45.2" "tokenizers==0.20.3" "datasets>=2.20,<4" '
   '"accelerate==0.34.2" "sentence-transformers==3.0.1" evaluate sacrebleu rouge_score nltk '
   'sentencepiece einops tenacity openai')
sh("pip uninstall -y -q torchvision")
sh(f"pip install -q -e {V2T_DIR} --no-deps")
sh("python -c \"import nltk; nltk.download('punkt', quiet=True); nltk.download('punkt_tab', quiet=True)\"")
sh("python -c \"import torch, transformers, datasets, vec2text; print('torch', torch.__version__, '| transformers', transformers.__version__, '| datasets', datasets.__version__, '| cuda', torch.cuda.is_available())\"")

## Inputs

The files are found by searching `/kaggle/input`, so this works whatever the dataset's mount path is,
and whether or not Kaggle unzipped the `.npz`.

The Llama-2 tokenizer and config are copied to `/kaggle/working/meta-llama/Llama-2-7b-hf`. Every
command then runs from `/kaggle/working`, so the model name `meta-llama/Llama-2-7b-hf` resolves to
that local folder, and the gated Hugging Face repo is never contacted.

In [ ]:
def find(name):
    return sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))


os.makedirs(NPY_DIR, exist_ok=True)
npz = [p for p in find("llama2_dolly15k.npz") if os.path.isfile(p) and zipfile.is_zipfile(p)]
if npz:
    print("extracting", npz[0])
    with zipfile.ZipFile(npz[0]) as z:
        for member in ("instructions.npy", "vectors.npy"):
            if not os.path.exists(f"{NPY_DIR}/{member}"):
                z.extract(member, NPY_DIR)
else:  # Kaggle already unzipped it
    for member in ("instructions.npy", "vectors.npy"):
        src = find(member)[0]
        print("linking", src)
        if not os.path.exists(f"{NPY_DIR}/{member}"):
            os.symlink(src, f"{NPY_DIR}/{member}")

tok_dir = os.path.dirname(find("tokenizer.model")[0])
llama_dir = f"{WORK}/meta-llama/Llama-2-7b-hf"
os.makedirs(llama_dir, exist_ok=True)
for name in ("config.json", "generation_config.json", "special_tokens_map.json",
             "tokenizer_config.json", "tokenizer.json", "tokenizer.model"):
    shutil.copy(f"{tok_dir}/{name}", llama_dir)

sh(f"ls -la {NPY_DIR} {llama_dir}")

## Build the training dataset

`build_dolly_llama2_dataset.py` does the following:
- tokenizes each instruction with the repo's own `tokenize_function`
- drops the instructions longer than 64 T5 tokens
- attaches each log-prob vector as the `frozen_embeddings` column
- splits the data 95/5 into train and validation (seed 42)

In [ ]:
if not os.path.exists(DATASET_DIR):
    sh(f"python {V2T_DIR}/experiment_scripts/logits/build_dolly_llama2_dataset.py "
       f"--data_dir {NPY_DIR} --out {DATASET_DIR}", cwd=WORK)
else:
    print("dataset already built:", DATASET_DIR)

## Build the one-million-instructions dataset

This uses the same build script and length rule as for Dolly, with 1% held out for validation.
The datasets cache goes to `/tmp` and is deleted afterwards, which frees about 13 GB of disk.

In [ ]:
OM_DIR = os.path.dirname(find("one_million_llama2/vectors.npy")[0])
print("vectors from", OM_DIR)
print(json.dumps(json.load(open(f"{OM_DIR}/verify.json")), indent=1))
if not os.path.exists(OM_DATASET_DIR):
    env = dict(os.environ, HF_DATASETS_CACHE="/tmp/hf_datasets_cache")
    sh(f"python {V2T_DIR}/experiment_scripts/logits/build_dolly_llama2_dataset.py "
       f"--data_dir {OM_DIR} --out {OM_DATASET_DIR} --test_size 0.01", cwd=WORK, env=env)
    shutil.rmtree("/tmp/hf_datasets_cache", ignore_errors=True)
sh("df -h / /tmp /kaggle/working | sort -u; free -g")

## Train

These are the same settings as the Dolly run, except:
- `--max_steps`, so the run fits the time limit;
- warmup over 1000 steps;
- eval and checkpoints every 1500 steps;
- the best checkpoint is chosen by BLEU.

In [ ]:
train_args = {
    "experiment": "inversion_from_logits_emb",
    "model_name_or_path": "t5-base",
    "embedder_model_name": "meta-llama/Llama-2-7b-hf",
    "embedder_torch_dtype": "float16",
    "dataset_name": "one_million_instructions",
    "max_seq_length": 64,
    "use_frozen_embeddings_as_input": 1,
    "mock_embedder": 1,
    "per_device_train_batch_size": 32,
    "per_device_eval_batch_size": 32,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "constant_with_warmup",
    "warmup_steps": 1000,
    "num_train_epochs": 100,    # upper bound; max_steps stops training first
    "max_steps": MAX_STEPS,
    "eval_steps": 1500,
    "save_steps": 1500,
    "logging_steps": 50,
    "save_total_limit": 2,      # plus the best checkpoint (by eval BLEU)
    "max_eval_samples": 500,
    "use_wandb": 0,
    "output_dir": OUT_DIR,
}
if SMOKE:
    train_args.update(warmup_steps=10, eval_steps=30, save_steps=30, logging_steps=5,
                      max_eval_samples=32)

env = dict(os.environ,
           CUDA_VISIBLE_DEVICES="0",
           PYTHONUNBUFFERED="1",
           TOKENIZERS_PARALLELISM="false",
           VEC2TEXT_TRAIN_DATASET_PATH=OM_DATASET_DIR,
           VEC2TEXT_EMBEDDER_NO_WEIGHTS="1",
           VEC2TEXT_SKIP_STANDARD_VAL="1",
           VEC2TEXT_BEST_METRIC="bleu_score")
cmd = f"python {V2T_DIR}/vec2text/run.py " + " ".join(f"--{k} {v}" for k, v in train_args.items())
sh(cmd, cwd=WORK, env=env, log=f"{WORK}/train_1m.log")

## Evaluate

Three models are scored on both validation sets, with three ways of generating the text:
- the best checkpoint by BLEU (`best_1m`)
- the final checkpoint (`final_1m`)
- the previous 14k-prompt Dolly model at epoch 40 (`dolly_ep40`)

In [ ]:
import glob as _glob

state = json.load(open(f"{OUT_DIR}/trainer_state.json"))
print("best checkpoint:", state.get("best_model_checkpoint"), "BLEU", state.get("best_metric"))
last_ckpt = sorted(_glob.glob(f"{OUT_DIR}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))[-1]
dolly_ep40 = os.path.dirname(find("checkpoint-17520/model.safetensors")[0])
models = f"--model best_1m={OUT_DIR} --model final_1m={last_ckpt} --model dolly_ep40={dolly_ep40}"
env = dict(os.environ, PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
for name, ds_dir in [("om_val", OM_DATASET_DIR), ("dolly_val", DATASET_DIR)]:
    sh(f"python {V2T_DIR}/experiment_scripts/logits/eval_inverter.py --dataset_dir {ds_dir} "
       f"--out_dir {EVAL_DIR}/{name} --n_examples 25 {models}"
       + (" --limit 64" if SMOKE else ""),
       cwd=WORK, env=env, log=f"{EVAL_DIR}_{name}.log")

## Results

In [ ]:
import pandas as pd

rows = []
for name in ("om_val", "dolly_val"):
    res = json.load(open(f"{EVAL_DIR}/{name}/results.json"))
    for model, r in res.items():
        for dec in ("greedy", "greedy_norep3", "beam4_norep3"):
            m = r[dec]
            rows.append({"eval_set": name, "model": model, "decoding": dec,
                         "eval_loss": round(r["loss"], 3), "BLEU": round(m["bleu_score"], 2),
                         "BLEU_sem": round(m["bleu_score_sem"], 2),
                         "token_F1": round(100 * m["token_set_f1"], 1),
                         "exact_%": round(100 * m["exact_match"], 1),
                         "ROUGE": round(100 * m["rouge_score"], 1)})
table = pd.DataFrame(rows)
table.to_csv(f"{EVAL_DIR}/summary.csv", index=False)
print(table.to_string(index=False))

print("\nper-eval history (eval set = held-out 1M prompts, 500 samples):")
for e in state["log_history"]:
    if "eval_one_million_instructions_loss" in e:
        print(f"step {e['step']:6d}  loss {e['eval_one_million_instructions_loss']:.3f}  "
              f"BLEU {e['eval_one_million_instructions_bleu_score']:.2f}  "
              f"F1 {e['eval_one_million_instructions_token_set_f1']:.3f}  "
              f"exact {e['eval_one_million_instructions_exact_match']:.3f}")

if SMOKE:  # keep logs and tables, drop the smoke checkpoints from the saved output
    shutil.rmtree(OUT_DIR)